# Experiment 15: Per-Question Heatmap and Ensemble Scatter Plot

Generates per-question QWK heatmaps across 40 prompts and 5-seed ensemble scatter plots for final evaluation.

## 0. Environment Setup

In [ ]:
import sys
sys.stdout.reconfigure(encoding='utf-8')
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score, mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr

REPO_ROOT = r"d:\Documents\mind-palace\03_Projects\Active\PROJ_2026_IndoASAG\IndoASAG"
DATASET_DIR = os.path.join(REPO_ROOT, "dataset")
PREDS_DIR   = os.path.join(REPO_ROOT, "results", "predictions")
FIGS_DIR    = os.path.join(REPO_ROOT, "results", "figures")
LATEX_FIGS_DIR = r"d:\Documents\mind-palace\03_Projects\Active\PROJ_2026_IndoASAG\latex\results\figures"

os.makedirs(FIGS_DIR, exist_ok=True)
os.makedirs(LATEX_FIGS_DIR, exist_ok=True)

matplotlib.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 9,
    "ytick.labelsize": 10,
    "figure.dpi": 150
})

def calculate_qwk(y_true, y_pred, thresholds=[0.2, 0.4, 0.6, 0.8]):
    y_t_d = np.digitize(y_true, thresholds)
    y_p_d = np.digitize(y_pred, thresholds)
    if len(np.unique(y_t_d)) < 2 and len(np.unique(y_p_d)) < 2:
        return 1.0 if np.array_equal(y_t_d, y_p_d) else 0.0
    try:
        return cohen_kappa_score(y_t_d, y_p_d, weights="quadratic")
    except Exception:
        return np.nan

## 1. Per-Question QWK Heatmap

In [ ]:
print("[1/2] Generating Heatmap per Question...")
train_df = pd.read_parquet(os.path.join(DATASET_DIR, "train.parquet"))
y_true_train = train_df["score_norm"].values

train_df_hm = train_df.copy()
train_df_hm["q_key"] = train_df_hm["topic"].astype(str) + "_" + train_df_hm["question_id"].astype(str)
unique_qs = sorted(train_df_hm["q_key"].unique())

models_oof = {
    "IndoBERT": "exp2_indobert_oof",
    "HC Morph SVR": "exp3_hc_oof",
    "Late Fusion Ridge": "exp5_late_fusion_ridge_oof",
}

SEEDS = [42, 100, 2026, 777, 999]
heatmap_data = []

for model_label, prefix in models_oof.items():
    # Calculate mean OOF predictions across 5 seeds
    seed_preds = []
    for s in SEEDS:
        p_path = os.path.join(PREDS_DIR, f"{prefix}_seed{s}.npy")
        if os.path.exists(p_path):
            seed_preds.append(np.load(p_path))
    
    if seed_preds:
        mean_preds = np.mean(seed_preds, axis=0)
    else:
        mean_preds = np.load(os.path.join(PREDS_DIR, f"{prefix}_seed42.npy"))
    
    row = {"Model": model_label}
    for q_key in unique_qs:
        mask = train_df_hm["q_key"] == q_key
        y_q = y_true_train[mask]
        p_q = mean_preds[mask]
        if len(y_q) >= 5:
            qwk_val = calculate_qwk(y_q, p_q)
            row[q_key] = qwk_val
        else:
            row[q_key] = np.nan
    heatmap_data.append(row)

hm_df = pd.DataFrame(heatmap_data).set_index("Model")

fig, ax = plt.subplots(figsize=(17, 4.2))
sns.heatmap(
    hm_df, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax,
    linewidths=0.6, linecolor="white", vmin=0.0, vmax=1.0,
    cbar_kws={'label': 'Quadratic Weighted Kappa (QWK)', 'shrink': 0.85},
    annot_kws={'fontsize': 8, 'fontweight': 'semibold'}
)

# Add vertical divider lines between topics (every 10 questions)
for div in [10, 20, 30]:
    ax.axvline(div, color="#2B5C8F", linewidth=2.0, linestyle="-")

ax.set_xlabel("Question Prompt ID Across 4 Domain Topics (Lifestyle, Sports, Politics, Tech)", fontsize=11)
ax.set_ylabel("Model Architecture", fontsize=11)
ax.set_title("Per-Question Grading Performance (QWK) across 40 Questions (IndoBERT vs. HC SVR vs. Late Fusion Ridge)", fontsize=12, fontweight="bold", pad=12)
plt.xticks(rotation=45, ha="right", fontsize=8.5)
plt.yticks(rotation=0, fontsize=10, fontweight="semibold")
plt.tight_layout()

hm_out_1 = os.path.join(FIGS_DIR, "heatmap_per_question_qwk.png")
hm_out_2 = os.path.join(LATEX_FIGS_DIR, "heatmap_per_question_qwk.png")
plt.savefig(hm_out_1, dpi=300, bbox_inches="tight")
plt.savefig(hm_out_2, dpi=300, bbox_inches="tight")
plt.close()
print(f"[OK] Heatmap saved to:\n  - {hm_out_1}\n  - {hm_out_2}")

## 2. Predicted vs. Actual Ensemble Scatter Plot

In [ ]:
print("[2/2] Regenerating Scatter Plot with explicit Ensemble label...")
test_df = pd.read_parquet(os.path.join(DATASET_DIR, "test.parquet"))
y_test  = test_df["score_norm"].values

ridge_seed_preds = []
for s in SEEDS:
    path = os.path.join(PREDS_DIR, f"exp5_late_fusion_ridge_test_seed{s}.npy")
    if os.path.exists(path):
        ridge_seed_preds.append(np.load(path))

y_pred_ensemble = np.mean(ridge_seed_preds, axis=0)

qwk_ens = calculate_qwk(y_test, y_pred_ensemble)
pearson_ens, _ = pearsonr(y_test, y_pred_ensemble)
mae_ens = mean_absolute_error(y_test, y_pred_ensemble)
rmse_ens = np.sqrt(mean_squared_error(y_test, y_pred_ensemble))

fig, ax = plt.subplots(figsize=(6.2, 5.8))
ax.scatter(y_test, y_pred_ensemble, alpha=0.55, s=32, color="#2B5C8F", edgecolors="none", label=f"Test Samples (N = {len(y_test)})")
ax.plot([0, 1], [0, 1], color="#D9534F", linestyle="--", linewidth=1.8, label="Ideal Prediction (y = x)")

annot_text = (
    f"5-Seed Ensemble Average:\n"
    f"QWK = {qwk_ens:.3f}\n"
    f"Pearson r = {pearson_ens:.3f}\n"
    f"MAE = {mae_ens:.3f}\n"
    f"RMSE = {rmse_ens:.3f}"
)
ax.text(0.05, 0.95, annot_text, transform=ax.transAxes, fontsize=9.5,
        verticalalignment="top", bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#CCCCCC", alpha=0.92))

ax.set_xlabel("Actual Score (Normalized)", fontsize=11)
ax.set_ylabel("Predicted Score (Normalized)", fontsize=11)
ax.set_title("Late Fusion Ridge: Predicted vs. Actual Scores", fontsize=12, fontweight="bold", pad=12)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="lower right", fontsize=8.5, framealpha=0.92)
ax.grid(True, linestyle="--", alpha=0.3)
sns.despine(ax=ax)
plt.tight_layout()

sc_out_1 = os.path.join(FIGS_DIR, "scatter_pred_vs_actual.png")
sc_out_2 = os.path.join(LATEX_FIGS_DIR, "scatter_pred_vs_actual.png")
plt.savefig(sc_out_1, dpi=300, bbox_inches="tight")
plt.savefig(sc_out_2, dpi=300, bbox_inches="tight")
plt.close()
print(f"[OK] Scatter plot saved to:\n  - {sc_out_1}\n  - {sc_out_2}")
print("Done!")